# BOLD SpO2-only occult-hypoxemia baseline diagnostic

## tl;dr

The locked SpO2-only OpenOx baseline transported substantially better to BOLD than the D028 compact model. On the same 11,880-pair denominator, the baseline predicted 4.38% risk versus 5.65% observed; D028 predicted 21.10%. Baseline Brier score was 0.05226 versus 0.12690, log loss 0.21301 versus 0.42119, PR-AUC 0.0998 versus 0.0735, and ROC-AUC 0.6608 versus 0.5680.

This is a post-validation diagnostic comparator specified after the D028 BOLD result was known. It indicates that the added compact predictors materially contributed to poor BOLD transport, but it is not a second confirmatory validation or a rescue-model selection exercise.


## Context & Methods

### Key assumptions

- The development cohort remains the frozen 180-second OpenOx cohort restricted to SpO2 92-96%, with SaO2 below 88% as the target.
- The candidate is the already-authorized SpO2-only ridge baseline from Notebook 13; no new predictors or model classes are searched.
- The final penalty is selected solely from the 250 pre-existing frozen OpenOx baseline tuning contexts using minimum mean inner log loss, mean Brier score, then smaller `C`.
- The model, coefficients, scoring specification, and diagnostic lock are written before this run loads BOLD SaO2 or prior D028 BOLD predictions.
- BOLD uses the same 11,880-pair denominator and 11,441 participants as D030-D031.
- Comparisons use paired 1,000-replicate participant-cluster bootstrap resampling.
- Because BOLD outcomes were already known when this diagnostic was authorized, all findings are exploratory mechanism evidence rather than confirmatory external validation.


In [ ]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import json
import pandas as pd

from bold_spo2_baseline_validation import run_validation
from qa_bold_spo2_baseline_validation import run_qa

ROOT = Path.cwd()
RESULTS = ROOT / "bold_spo2_baseline_validation"


## Data

### 1. Recreate the OpenOx-only lock and score BOLD unchanged

Running this cell recreates all model, prediction, metric, bootstrap, chronology, and hash artifacts. The OpenOx-only lock is persisted before external outcome access within the run.


In [ ]:
summary = run_validation()
pd.Series(summary, name="value").to_frame()


### 2. Verify the frozen penalty selection and model contract

In [ ]:
penalty = pd.read_csv(RESULTS / "baseline_penalty_selection.csv")
coefficients = pd.read_csv(RESULTS / "baseline_coefficients.csv")
lock = json.loads((RESULTS / "baseline_model_lock.json").read_text())
penalty, coefficients, pd.Series(lock, name="value").to_frame()


## Results

### 3. Compare unchanged BOLD performance on the identical denominator

In [ ]:
comparison = pd.read_csv(RESULTS / "bold_baseline_vs_compact.csv")
headline = [
    "mean_predicted", "calibration_intercept", "calibration_slope",
    "brier", "log_loss", "pr_auc", "roc_auc",
    "sensitivity_5pct", "specificity_5pct", "ppv_5pct", "npv_5pct",
    "flagged_rate_5pct", "net_benefit_5pct",
]
comparison.loc[comparison["metric"].isin(headline)].reset_index(drop=True)


### 4. Inspect participant-bootstrap uncertainty and SpO2-specific calibration

In [ ]:
intervals = pd.read_csv(RESULTS / "bold_baseline_bootstrap_intervals.csv")
differences = pd.read_csv(
    RESULTS / "bold_baseline_vs_compact_bootstrap_differences.csv"
)
calibration = pd.read_csv(RESULTS / "bold_baseline_calibration_by_spo2.csv")
metrics = [
    "mean_predicted", "calibration_intercept", "calibration_slope",
    "brier", "log_loss", "pr_auc", "roc_auc",
]
(
    intervals.loc[intervals["metric"].isin(metrics)].reset_index(drop=True),
    differences.loc[differences["metric"].isin(metrics)].reset_index(drop=True),
    calibration,
)


### 5. Run independent metric and artifact QA

In [ ]:
qa_result = run_qa()
qa = pd.read_csv(RESULTS / "bold_baseline_independent_qa.csv")
qa_result, qa


## Takeaways

- The SpO2-only baseline is much closer to BOLD's observed event rate: 4.38% predicted versus 5.65% observed. Its calibration intercept is 0.281 and slope is 0.611. Calibration is not perfect—the model underpredicts on average and remains overfit—but it is far better than D028's 21.10% prediction, -2.114 intercept, and 0.119 slope.
- Probability loss and ranking both improve materially: baseline-minus-D028 Brier is -0.07464, log loss -0.20819, PR-AUC +0.0262, and ROC-AUC +0.0928. The paired participant-bootstrap intervals exclude zero for all four differences.
- The result supports a specific interpretation: the added age, sex, heart-rate, and respiratory-rate terms—under major ICU case-mix and measurement-timing shift—substantially degraded transport relative to the simpler SpO2 signal.
- The baseline is still not clinically validated or deployment-ready. Its slope is below one, its overall prediction is low, and its 5% threshold exchanges lower sensitivity for much higher specificity. The exercise diagnoses D028's failure; it does not authorize choosing a new clinical model after seeing BOLD.
- ENCoDE remains unscorable for occult-risk models because its eligible denominator contains zero events.
